In [12]:
%pip install -q mysql-connector-python

from pathlib import Path
import os
import pandas as pd
import mysql.connector

DATA_DIR_CANDIDATES = [
    Path.cwd() / 'Dataset',
    Path.cwd().parent / 'Dataset',
    Path.cwd(),
    Path.cwd().parent,
]
DATA_DIR = next(
    (path for path in DATA_DIR_CANDIDATES if (path / 'movies.csv').exists()),
    None,
)
FILES = ['movies', 'cast', 'crew', 'genres', 'movie_genres', 'movie_keywords']
PLACEHOLDERS = {'', 'null', 'none', 'nan', 'n/a', 'na', '\\n'}
MYSQL_CONFIG = {
    'host': os.getenv('MYSQL_HOST', 'localhost'),
    'port': int(os.getenv('MYSQL_PORT', '3306')),
    'user': os.getenv('MYSQL_USER', 'root'),
    'password': os.getenv('MYSQL_PASSWORD', 'root'),
    'database': os.getenv('MYSQL_DATABASE', 'TMDB'),
}
if DATA_DIR is None:
    searched_paths = ', '.join(str(path.resolve()) for path in DATA_DIR_CANDIDATES)
    raise FileNotFoundError(f'CSV directory not found. Searched: {searched_paths}')
print(f'Data directory: {DATA_DIR.resolve()}')
print(f'MySQL target: {MYSQL_CONFIG["host"]}:{MYSQL_CONFIG["port"]}/{MYSQL_CONFIG["database"]}')

Note: you may need to restart the kernel to use updated packages.
Data directory: D:\Guvi\TMDB_Movie_Analytics\Dataset
MySQL target: localhost:3306/TMDB



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [13]:
def load_csv(name):
    frame = pd.read_csv(DATA_DIR / f'{name}.csv', dtype='string', keep_default_na=False)
    for column in frame.columns:
        frame[column] = frame[column].str.strip()
        frame[column] = frame[column].mask(frame[column].str.lower().isin(PLACEHOLDERS))
    return frame.drop_duplicates().reset_index(drop=True)

def to_int(frame, columns):
    for column in columns:
        frame[column] = pd.to_numeric(frame[column], errors='coerce').round().astype('Int64')
    return frame

def to_float(frame, columns):
    for column in columns:
        frame[column] = pd.to_numeric(frame[column], errors='coerce').astype('Float64')
    return frame

raw = {name: load_csv(name) for name in FILES}

movies = raw['movies'][['movie_id', 'title', 'original_language', 'release_date', 'runtime', 'budget', 'revenue', 'popularity', 'vote_average', 'vote_count', 'status', 'overview']].copy()
movies = to_int(movies, ['movie_id', 'runtime', 'budget', 'revenue', 'vote_count'])
movies = to_float(movies, ['popularity', 'vote_average'])
movies['release_date'] = pd.to_datetime(movies['release_date'], errors='coerce').dt.date
movies = movies.dropna(subset=['movie_id']).drop_duplicates('movie_id').copy()
movies['movie_id'] = movies['movie_id'].astype('int64')
unknown_budget_count = int((movies['budget'] == 0).sum())
unknown_revenue_count = int((movies['revenue'] == 0).sum())
unknown_rating_count = int((movies['vote_average'] == 0).sum())
movies[['budget', 'revenue', 'vote_average']] = movies[['budget', 'revenue', 'vote_average']].replace(0, pd.NA)
for column, default, size in [('title', 'Unknown title', 255), ('original_language', 'und', 10), ('status', 'Unknown', 50), ('overview', '', None)]:
    movies[column] = movies[column].fillna(default)
    if size:
        movies[column] = movies[column].str.slice(0, size)
movies['runtime'] = movies['runtime'].fillna(0).astype('int64')
movies['vote_count'] = movies['vote_count'].fillna(0).astype('int64')
for column in ['budget', 'revenue']:
    movies[column] = movies[column].astype('Int64')
for column in ['popularity', 'vote_average']:
    movies[column] = movies[column].astype('Float64')

genres = to_int(raw['genres'].copy(), ['genre_id']).dropna(subset=['genre_id']).drop_duplicates('genre_id').copy()
genres['genre_id'] = genres['genre_id'].astype('int64')
genres['genre_name'] = genres['genre_name'].fillna('Unknown').str.slice(0, 255)

def clean_detail(name, key, text_columns):
    frame = raw[name].copy()
    integer_columns = [column for column in frame.columns if column.endswith('_id') or column == 'cast_order']
    frame = to_int(frame, integer_columns).dropna(subset=[key, 'movie_id']).drop_duplicates(key).copy()
    frame = frame[frame['movie_id'].isin(movies['movie_id'])].copy()
    for column in integer_columns:
        frame[column] = frame[column].fillna(0).astype('int64')
    for column in text_columns:
        frame[column] = frame[column].fillna('Unknown').str.slice(0, 255)
    return frame

cast = clean_detail('cast', 'cast_row_id', ['actor_name', 'character_name'])
crew = clean_detail('crew', 'crew_row_id', ['person_name', 'job', 'department'])
movie_genres = to_int(raw['movie_genres'].copy(), ['movie_id', 'genre_id']).dropna().drop_duplicates(['movie_id', 'genre_id'])
movie_genres = movie_genres[movie_genres['movie_id'].isin(movies['movie_id']) & movie_genres['genre_id'].isin(genres['genre_id'])].astype('int64')
movie_keywords = to_int(raw['movie_keywords'].copy(), ['movie_id', 'keyword_id']).dropna(subset=['movie_id', 'keyword_id']).drop_duplicates(['movie_id', 'keyword_id'])
movie_keywords = movie_keywords[movie_keywords['movie_id'].isin(movies['movie_id'])].copy()
movie_keywords[['movie_id', 'keyword_id']] = movie_keywords[['movie_id', 'keyword_id']].astype('int64')
movie_keywords['keyword_name'] = movie_keywords['keyword_name'].fillna('Unknown').str.slice(0, 255)
cleaned = {'movies': movies, 'genres': genres, 'cast': cast, 'crew': crew, 'movie_genres': movie_genres, 'movie_keywords': movie_keywords}
cleaning_report = pd.DataFrame({'source_rows_after_exact_dedup': {name: len(raw[name]) for name in FILES}, 'cleaned_rows': {name: len(cleaned[name]) for name in FILES}})
print(f'Converted {unknown_budget_count} unknown budgets, {unknown_revenue_count} unknown revenues, and {unknown_rating_count} unknown ratings from 0 to SQL NULL.')
cleaning_report

Converted 90 unknown budgets, 102 unknown revenues, and 0 unknown ratings from 0 to SQL NULL.


,source_rows_after_exact_dedup,cleaned_rows
movies,2503,2503
cast,24902,24902
crew,7289,7289
genres,19,19
movie_genres,6878,6878
movie_keywords,41681,41681


In [14]:
MOVIE_COLUMNS = ['movie_id', 'title', 'original_language', 'release_date', 'runtime', 'budget', 'revenue', 'popularity', 'vote_average', 'vote_count', 'status', 'overview']
assert list(movies.columns) == MOVIE_COLUMNS
assert movies['movie_id'].is_unique and genres['genre_id'].is_unique
assert cast['movie_id'].isin(movies['movie_id']).all()
assert crew['movie_id'].isin(movies['movie_id']).all()
assert movie_genres['movie_id'].isin(movies['movie_id']).all() and movie_genres['genre_id'].isin(genres['genre_id']).all()
assert movie_keywords['movie_id'].isin(movies['movie_id']).all()
assert movie_keywords[['movie_id', 'keyword_id']].duplicated().sum() == 0
assert movie_keywords['keyword_id'].dtype == 'int64'
assert movies['release_date'].dtype == 'object'
print('Validation passed: all cleaning rules, composite keys, and nullable analysis fields are valid.')

Validation passed: all cleaning rules, composite keys, and nullable analysis fields are valid.


In [15]:
audit = {
    'exact_duplicate_rows': sum(frame.duplicated().sum() for frame in cleaned.values()),
    'movies_duplicate_ids': int(movies['movie_id'].duplicated().sum()),
    'genres_duplicate_ids': int(genres['genre_id'].duplicated().sum()),
    'cast_orphans': int((~cast['movie_id'].isin(movies['movie_id'])).sum()),
    'crew_orphans': int((~crew['movie_id'].isin(movies['movie_id'])).sum()),
    'movie_genres_movie_orphans': int((~movie_genres['movie_id'].isin(movies['movie_id'])).sum()),
    'movie_genres_genre_orphans': int((~movie_genres['genre_id'].isin(genres['genre_id'])).sum()),
    'movie_keywords_orphans': int((~movie_keywords['movie_id'].isin(movies['movie_id'])).sum()),
    'movie_keywords_duplicate_pairs': int(movie_keywords[['movie_id', 'keyword_id']].duplicated().sum()),
    'invalid_vote_average': int((movies['vote_average'].dropna().between(0, 10) == False).sum()),
    'negative_vote_count': int((movies['vote_count'] < 0).sum()),
    'negative_runtime': int((movies['runtime'] < 0).sum()),
    'budget_unknown_sql_nulls': int(movies['budget'].isna().sum()),
    'revenue_unknown_sql_nulls': int(movies['revenue'].isna().sum()),
    'rating_unknown_sql_nulls': int(movies['vote_average'].isna().sum()),
}
assert all(value == 0 for key, value in audit.items() if key not in {'budget_unknown_sql_nulls', 'revenue_unknown_sql_nulls', 'rating_unknown_sql_nulls'})
print('Complete cleaning audit passed.')
pd.Series(audit, name='count')

Complete cleaning audit passed.


exact_duplicate_rows                0
movies_duplicate_ids                0
genres_duplicate_ids                0
cast_orphans                        0
crew_orphans                        0
movie_genres_movie_orphans          0
movie_genres_genre_orphans          0
movie_keywords_orphans              0
movie_keywords_duplicate_pairs      0
invalid_vote_average                0
negative_vote_count                 0
negative_runtime                    0
budget_unknown_sql_nulls           90
revenue_unknown_sql_nulls         102
rating_unknown_sql_nulls            0
Name: count, dtype: int64

In [16]:
server_config = MYSQL_CONFIG.copy()
database_name = server_config.pop('database')
server_connection = mysql.connector.connect(**server_config)
server_cursor = server_connection.cursor()
server_cursor.execute(f'CREATE DATABASE IF NOT EXISTS `{database_name}` CHARACTER SET utf8mb4 COLLATE utf8mb4_unicode_ci')
server_connection.commit()
server_cursor.close()
server_connection.close()

connection = mysql.connector.connect(**MYSQL_CONFIG)
cursor = connection.cursor()
DDL = [
    'DROP TABLE IF EXISTS movie_keywords', 'DROP TABLE IF EXISTS movie_genres', 'DROP TABLE IF EXISTS `cast`', 'DROP TABLE IF EXISTS crew', 'DROP TABLE IF EXISTS genres', 'DROP TABLE IF EXISTS movies',
    'CREATE TABLE movies (movie_id INT PRIMARY KEY, title VARCHAR(255) NOT NULL, original_language VARCHAR(10) NOT NULL, release_date DATE, runtime INT, budget BIGINT, revenue BIGINT, popularity FLOAT, vote_average FLOAT, vote_count INT, status VARCHAR(50), overview TEXT) ENGINE=InnoDB',
    'CREATE TABLE genres (genre_id INT PRIMARY KEY, genre_name VARCHAR(255) NOT NULL) ENGINE=InnoDB',
    'CREATE TABLE `cast` (cast_row_id INT PRIMARY KEY, movie_id INT NOT NULL, person_id INT, actor_name VARCHAR(255), character_name VARCHAR(255), cast_order INT, FOREIGN KEY (movie_id) REFERENCES movies(movie_id)) ENGINE=InnoDB',
    'CREATE TABLE crew (crew_row_id INT PRIMARY KEY, movie_id INT NOT NULL, person_id INT, person_name VARCHAR(255), job VARCHAR(255), department VARCHAR(255), FOREIGN KEY (movie_id) REFERENCES movies(movie_id)) ENGINE=InnoDB',
    'CREATE TABLE movie_genres (movie_id INT NOT NULL, genre_id INT NOT NULL, PRIMARY KEY (movie_id, genre_id), CONSTRAINT fk_movie_genres_movie FOREIGN KEY (movie_id) REFERENCES movies(movie_id), CONSTRAINT fk_movie_genres_genre FOREIGN KEY (genre_id) REFERENCES genres(genre_id)) ENGINE=InnoDB',
    'CREATE TABLE movie_keywords (movie_id INT NOT NULL, keyword_id INT NOT NULL, keyword_name VARCHAR(255) NOT NULL, PRIMARY KEY (movie_id, keyword_id), CONSTRAINT fk_movie_keywords_movie FOREIGN KEY (movie_id) REFERENCES movies(movie_id)) ENGINE=InnoDB',
]
TABLE_COLUMNS = {
    'movies': MOVIE_COLUMNS, 'genres': ['genre_id', 'genre_name'],
    'cast': ['cast_row_id', 'movie_id', 'person_id', 'actor_name', 'character_name', 'cast_order'],
    'crew': ['crew_row_id', 'movie_id', 'person_id', 'person_name', 'job', 'department'],
    'movie_genres': ['movie_id', 'genre_id'], 'movie_keywords': ['movie_id', 'keyword_id', 'keyword_name'],
}

def mysql_records(frame, columns):
    records = []
    for row in frame[columns].itertuples(index=False, name=None):
        records.append(tuple(None if pd.isna(value) else value.item() if hasattr(value, 'item') else value for value in row))
    return records

for statement in DDL:
    cursor.execute(statement)
for table_name, columns in TABLE_COLUMNS.items():
    placeholders = ', '.join(['%s'] * len(columns))
    column_list = ', '.join(f'`{column}`' for column in columns)
    cursor.executemany(f'INSERT INTO `{table_name}` ({column_list}) VALUES ({placeholders})', mysql_records(cleaned[table_name], columns))
connection.commit()
for table_name in TABLE_COLUMNS:
    cursor.execute(f'SELECT COUNT(*) FROM `{table_name}`')
    print(table_name, cursor.fetchone()[0])
cursor.close()
connection.close()
print(f'Database `{database_name}` and all six tables loaded successfully.')

movies 2503
genres 19
cast 24902
crew 7289
movie_genres 6878
movie_keywords 41681
Database `TMDB` and all six tables loaded successfully.
